# 03 - Avaliacao: Metricas + Tabelas + Visualizacoes

Este notebook:
1. Clona o repositorio do projeto
2. Carrega todas as meshes (ground truth, V2, V1 naive)
3. Amostra 100K points de cada mesh
4. Calcula metricas: CD, ECD, NC, #V, #F, V_Ratio, F_Ratio
5. Gera tabela comparativa (CSV)
6. Gera visualizacoes

In [ ]:
# Cell 1 - Clonar repositorio e instalar dependencias
!git clone https://github.com/matheus2049alves/cg_mesh.git
%cd /content/cg_mesh

!pip install -q -r requirements.txt

In [ ]:
# Cell 2 - Setup de paths e imports
import os
import numpy as np
import trimesh
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import glob

PROJECT_ROOT = os.path.abspath(".")
MESHES_GT = os.path.join(PROJECT_ROOT, "data", "meshes", "ground_truth")
MESHES_V2 = os.path.join(PROJECT_ROOT, "data", "meshes", "meshanything_v2")
MESHES_V1 = os.path.join(PROJECT_ROOT, "data", "meshes", "baselines", "naive")
RESULTS_TABLES = os.path.join(PROJECT_ROOT, "results", "tables")
RESULTS_FIGURES = os.path.join(PROJECT_ROOT, "results", "figures")

os.makedirs(RESULTS_TABLES, exist_ok=True)
os.makedirs(RESULTS_FIGURES, exist_ok=True)

print("Paths configurados.")

In [ ]:
# Cell 3 - Carregar meshes
def load_meshes(mesh_dir):
    """Carrega todas as .obj de um diretorio."""
    meshes = {}
    for path in sorted(glob.glob(os.path.join(mesh_dir, "*.obj"))):
        name = os.path.splitext(os.path.basename(path))[0]
        meshes[name] = trimesh.load(path, force="mesh")
    return meshes

gt_meshes = load_meshes(MESHES_GT)
v2_meshes = load_meshes(MESHES_V2)
v1_meshes = load_meshes(MESHES_V1)

print(f"Ground truth: {len(gt_meshes)} meshes")
print(f"MeshAnything V2: {len(v2_meshes)} meshes")
print(f"Baseline V1: {len(v1_meshes)} meshes")

# Encontrar nomes em comum
common_names = sorted(set(gt_meshes.keys()) & set(v2_meshes.keys()) & set(v1_meshes.keys()))
print(f"\nMeshes para avaliar (presentes em todos): {len(common_names)}")
for name in common_names:
    print(f"  - {name}")

## Funcoes de Metricas

In [ ]:
# Cell 4 - Implementacao das metricas

def sample_points(mesh, n_samples=100_000):
    """Amostra pontos uniformemente da superficie da mesh."""
    points, face_idx = mesh.sample(n_samples, return_index=True)
    normals = mesh.face_normals[face_idx]
    return points, normals


def sample_edge_points(mesh, n_samples=100_000, angle_threshold=np.pi/6):
    """Amostra pontos das arestas com angulo diedro > threshold."""
    edges = mesh.edges_sorted
    faces_adj = mesh.edges_face
    face_normals = mesh.face_normals

    sharp_edges = []
    for i, (f0, f1) in enumerate(faces_adj):
        if f0 == -1 or f1 == -1:
            continue
        n0 = face_normals[f0]
        n1 = face_normals[f1]
        cos_angle = np.dot(n0, n1) / (np.linalg.norm(n0) * np.linalg.norm(n1) + 1e-8)
        angle = np.arccos(np.clip(cos_angle, -1, 1))
        if angle > angle_threshold:
            sharp_edges.append(i)

    if len(sharp_edges) == 0:
        return sample_points(mesh, n_samples)

    verts = mesh.vertices
    edge_verts = edges[sharp_edges]

    t = np.random.uniform(0, 1, (n_samples, 1))
    edge_idx = np.random.choice(len(sharp_edges), n_samples)
    v0 = verts[edge_verts[edge_idx, 0]]
    v1 = verts[edge_verts[edge_idx, 1]]
    points = v0 * (1 - t) + v1 * t

    return points, None


def chamfer_distance(pts1, pts2):
    """Calcula Chamfer Distance bidirecional."""
    tree1 = cKDTree(pts1)
    tree2 = cKDTree(pts2)
    dist1, _ = tree2.query(pts1)
    dist2, _ = tree1.query(pts2)
    cd = np.mean(dist1) + np.mean(dist2)
    return cd


def normal_consistency(pts1, normals1, pts2, normals2, k=10):
    """Calcula Normal Consistency entre duas point clouds."""
    tree2 = cKDTree(pts2)
    _, idx2 = tree2.query(pts1, k=k)

    cos_sim = 0
    for i in range(len(pts1)):
        n1 = normals1[i]
        n2_neighbors = normals2[idx2[i]]
        n2_avg = np.mean(n2_neighbors, axis=0)
        cos = np.dot(n1, n2_avg) / (np.linalg.norm(n1) * np.linalg.norm(n2_avg) + 1e-8)
        cos_sim += np.abs(cos)

    nc = cos_sim / len(pts1)
    return nc


print("Funcoes de metricas definidas.")

In [ ]:
# Cell 5 - Calcular metricas para todas as meshes
N_SAMPLES = 100_000

results = []

for name in common_names:
    print(f"\nProcessando: {name}")

    gt = gt_meshes[name]
    v2 = v2_meshes[name]
    v1 = v1_meshes[name]

    gt_pts, gt_normals = sample_points(gt, N_SAMPLES)
    v2_pts, v2_normals = sample_points(v2, N_SAMPLES)
    v1_pts, v1_normals = sample_points(v1, N_SAMPLES)

    gt_n_verts = len(gt.vertices)
    gt_n_faces = len(gt.faces)

    for label, mesh, pts, normals in [
        ("meshanything_v2", v2, v2_pts, v2_normals),
        ("baseline_naive", v1, v1_pts, v1_normals),
    ]:
        n_verts = len(mesh.vertices)
        n_faces = len(mesh.faces)

        cd = chamfer_distance(gt_pts, pts)
        nc = normal_consistency(gt_pts, gt_normals, pts, normals)

        gt_edge_pts, _ = sample_edge_points(gt, min(N_SAMPLES, 50000))
        mesh_edge_pts, _ = sample_edge_points(mesh, min(N_SAMPLES, 50000))
        ecd = chamfer_distance(gt_edge_pts, mesh_edge_pts)

        v_ratio = n_verts / gt_n_verts
        f_ratio = n_faces / gt_n_faces

        results.append({
            "name": name,
            "method": label,
            "CD": cd * 100,
            "ECD": ecd * 100,
            "NC": nc,
            "n_verts": n_verts,
            "n_faces": n_faces,
            "V_Ratio": v_ratio,
            "F_Ratio": f_ratio,
        })

    print(f"  CD: V2={results[-2]['CD']:.4f}, V1={results[-1]['CD']:.4f}")
    print(f"  NC: V2={results[-2]['NC']:.4f}, V1={results[-1]['NC']:.4f}")

print(f"\nTotal de resultados: {len(results)}")

In [ ]:
# Cell 6 - Gerar tabela comparativa
df = pd.DataFrame(results)

csv_path = os.path.join(RESULTS_TABLES, "evaluation_full.csv")
df.to_csv(csv_path, index=False)
print(f"Tabela completa salva: {csv_path}")

df_avg = df.groupby("method").agg({
    "CD": "mean",
    "ECD": "mean",
    "NC": "mean",
    "n_verts": "mean",
    "n_faces": "mean",
    "V_Ratio": "mean",
    "F_Ratio": "mean",
}).round(4)

print("\n" + "=" * 60)
print("TABELA MEDIA POR METODO")
print("=" * 60)
print(df_avg.to_string())

csv_avg_path = os.path.join(RESULTS_TABLES, "evaluation_avg.csv")
df_avg.to_csv(csv_avg_path)
print(f"\nTabela media salva: {csv_avg_path}")

In [ ]:
# Cell 7 - Visualizacao: barras comparativas
metrics = ["CD", "ECD", "NC", "n_verts", "n_faces", "V_Ratio", "F_Ratio"]
methods = ["meshanything_v2", "baseline_naive"]
labels = ["MeshAnything V2", "Baseline V1"]
colors = ["#2196F3", "#FF9800"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]
    values = [df_avg.loc[m, metric] for m in methods]
    bars = ax.bar(labels, values, color=colors)
    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val:.4f}", ha="center", va="bottom", fontsize=10)

axes[-1].set_visible(False)

plt.suptitle("Comparacao: MeshAnything V2 vs Baseline V1", fontsize=16, fontweight="bold")
plt.tight_layout()

fig_path = os.path.join(RESULTS_FIGURES, "comparison_bars.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Grafico salvo: {fig_path}")
plt.show()

In [ ]:
# Cell 8 - Resumo final
print("=" * 60)
print("RESUMO - Passo 3: Avaliacao")
print("=" * 60)
print(f"\nMeshes avaliadas: {len(common_names)}")
print(f"Amostras por mesh: {N_SAMPLES:,} points")
print(f"\nTabelas salvas em: {RESULTS_TABLES}")
print(f"Visualizacoes em: {RESULTS_FIGURES}")
print(f"\nArquivos gerados:")
print(f"  - evaluation_full.csv (resultados por mesh)")
print(f"  - evaluation_avg.csv (medias por metodo)")
print(f"  - comparison_bars.png (grafico comparativo)")